## 0. 라이브러리 불러오기/CUDA 설정  

In [2]:
import torch
import json
import numpy as np
import pandas as pd
import re
import os
from datasets import Dataset, Audio # HuggingFace datasets 라이브러리
from transformers import (
    Wav2Vec2ForCTC,             # CTC 기반 음성인식 모델 (ASR)
    Wav2Vec2Processor,          # tokenizer + feature extractor 통합 객체
    Wav2Vec2CTCTokenizer,       # CTC용 문자 토크나이저
    Wav2Vec2FeatureExtractor,   # waveform → model input feature 변환
    Trainer,                    # HuggingFace 학습 루프 추상화 클래스
    TrainingArguments,          # 학습 하이퍼파라미터 설정 객체
    EarlyStoppingCallback       # 조기 종료 콜백 (overfitting 방지)
)
from dataclasses import dataclass   # 데이터 구조 정의용
from typing import Union
import evaluate
import librosa  # 오디오 처리 라이브러리
from praatio import tgio    # Praat TextGrid 파싱 라이브러리

ModuleNotFoundError: No module named 'datasets'

In [ ]:
# GPU 디바이스 설정
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "9"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"현재 사용 중인 디바이스: {device}")

현재 사용 중인 디바이스: cuda


## 1. 전처리 및 필터링 
[L2-ARCTIC: a non-native English speech corpus](https://psi.engr.tamu.edu/l2-arctic-corpus/)   
일반적으로 발음 오류 검출을 위해 `L2-ARCTIC`를 사용한다는 논문에 따라 발음 오류 검출을 위해 `L2-ARCTIC` 데이터를 파인튜닝에 사용하였습니다.   
> 현재 발음 오류 검출 분야에서 가장 널리 이용되고 있는 데이터셋은 L2-ARCTIC(Zhao et al., 2018)이다. L2-ARCTIC 데이터셋은 영어 학습자의 비원어민 발화 데이터셋이며 최신 버전은 힌디어, 한국어, 표준 중국어, 스페인어, 아랍어, 베트남어의 모어 배경을 가진 총 24명의 화자의 발화를 수록하고 있다. 그중 각 화자마다 150개 발화가 단어와 음소 단위로 각각 전사 및 정렬되어 발음 오류 음소에 대해서는 표준 음소(canonical phoneme), 실제 음소, 발음 오류 종류가 함께 레이블링되어 있다.
- 출처: 교차 언어적 파인튜닝을 사용한 wav2vec 2.0 기반 발음 오류 검출 (2022)

l2-artic 데이터 구조는 다음과 같습니다: 
+ `/wav` : 44.1kHz WAV 음성 파일
+ `/transcript` : 정서적(orthographic) 전사 텍스트
+ `/textgrid` : forced-alignment 기반 음소 전사
+ `/annotation` : 수동(manual) 음소 주석 TextGrid 파일
이번 파인튜닝에서는 발음 오류 정보가 포함된 수동 주석(annotation) 만을 활용하였습니다.

![img](./img/l2-arctic.webp)   
l2-artic의 textgrid는 다음과 같은 형식으로 있습니다.  
수동 주석 기반의 음소 단위 발음 데이터만 추출하여, 정제 및 무음 통합을 거친 후 CTC 학습용 음소 시퀀스로 구성하였습니다.
+ 20초 초과 음성 제거
    + 긴 음성은 학습 안정성 및 메모리 효율을 위해 제외
+ TextGrid의 `phones` tier 파싱
    + 각 구간의 음소 라벨 추출
    + `cpl`, `ppl` 구조 중 발음 학습자 발음(`ppl`)만 사용
+ 음소 정제
    + 대문자 통일
    + 숫자 제거 (stress 표기 제거)
    + 특수문자 제거
    + `ERR → ER` 등 예외 처리
+ 무음 토큰 통합
    + `SP, PAU, SIL, SPN` → `SIL` 로 통일

In [ ]:
def clean_phoneme(label):
    label = label.upper().strip()
    label = re.sub(r'[0-9]', '', label)
    label = re.sub(r'[^A-Z]', '', label)
    if label == "ERR": 
        label = "ER"
    return label

In [ ]:
def process_l2arctic(base_dir):
    dataset = []
    
    for speaker in os.listdir(base_dir):
        speaker_path = os.path.join(base_dir, speaker)

        if not os.path.isdir(speaker_path): continue
        if speaker == "suitcase_corpus": continue # 폴더 제외
        
        tg_dir = os.path.join(speaker_path, "annotation")
        wav_dir = os.path.join(speaker_path, "wav")
        if not os.path.exists(tg_dir): continue

        for tg_file in os.listdir(tg_dir):
            if not tg_file.endswith(".TextGrid"): continue
            file_id = tg_file.replace(".TextGrid", "")
            
            wav_path = os.path.join(wav_dir, f"{file_id}.wav")
            tg_path = os.path.join(tg_dir, tg_file)
            if not os.path.exists(wav_path): continue

            try:
                duration = librosa.get_duration(path=wav_path)
                if duration > 20.0: continue # 20초 초과 데이터 제거

                tg = tgio.openTextgrid(tg_path)
                ppl_list = []

                # Phones Tier 파싱
                for entry in tg.tierDict['phones'].entryList:
                    label = entry.label.strip()
                    if not label: continue
                        
                    parts = [p.strip() for p in label.split(',')]
                    raw_ppl = parts[1] if len(parts) > 1 else parts[0] # part[0]는 cpl
                    ppl = clean_phoneme(raw_ppl)
                    
                    # 무음 토큰 통일
                    noise_tokens = ['SP', 'PAU', 'SIL', 'SPN', '']
                    if ppl in noise_tokens: ppl = 'SIL'
                    
                    ppl_list.append(ppl)

                dataset.append({
                    "wav": wav_path, 
                    "duration": round(duration, 2), 
                    "speaker": speaker,
                    "ppl": " ".join(ppl_list)
                })
            except Exception as e: 
                print(f"Error: {speaker}/{file_id}: {e}")
    
    print(f"총 {len(dataset)}개 데이터 로드 완료")
    return dataset

In [ ]:
raw_data = process_l2arctic("../data") 
df = pd.DataFrame(raw_data)

총 3599개 데이터 로드 완료


In [ ]:
df

,wav,duration,speaker,ppl
0,../data/RRBI/wav/arctic_a0020.wav,4.12,RRBI,SIL K L AH P S SIL AH N SIL B AO L S SIL AE N ...
1,../data/RRBI/wav/arctic_a0024.wav,4.58,RRBI,SIL IH T W AH Z M AY R IY P AO R T S F R AH M ...
2,../data/RRBI/wav/arctic_a0029.wav,3.71,RRBI,SIL DH EH R F AO SIL S IH Z W EH R SIL AO R EH...
3,../data/RRBI/wav/arctic_a0121.wav,3.10,RRBI,SIL D EY SIL EY R T D IH N ER AE R T D AH F AH...
4,../data/RRBI/wav/arctic_a0073.wav,4.44,RRBI,SIL DH AH P R AH M OW T ER S AY Z V ER HH EH V...
...,...,...,...,...
3594,../data/HQTV/wav/arctic_a0111.wav,4.54,HQTV,SIL IH SIL S T EH D SIL HH IY JH OY Z SIL HH U...
3595,../data/HQTV/wav/arctic_b0165.wav,4.34,HQTV,SIL HH IY P UW R D SIL AE N D SIL AH L AO SIL ...
3596,../data/HQTV/wav/arctic_a0076.wav,3.16,HQTV,SIL D AH G R EY AY Z F AO N T AH SIL SIL D AH ...
3597,../data/HQTV/wav/arctic_b0304.wav,5.65,HQTV,SIL HH IH SIL V R IY V SIL AH S W AY F SIL SIL...


## 2. 음소 Vocab 만들기/토크나이저

### 1) Vocab 만들기
본 모델은 음소 단위 CTC 학습을 수행하므로, 텍스트가 아닌 phoneme 단위 vocabulary를 별도로 구축하였습니다.

In [ ]:
def build_vocab(dataframe):
    # ppl 컬럼의 모든 음소를 수집 (공백 기준 split)
    vocab_set = set()
    phoneme_lists = dataframe['ppl'].dropna().apply(lambda x: x.split())
    for phonemes in phoneme_lists:
        vocab_set.update(phonemes)
    
    # 정렬
    vocab_list = sorted(list(vocab_set))
    
    # 특수 토큰 정의 ([PAD]=0, [UNK]=1, |=2)
    vocab_dict = {
        "[PAD]": 0, # padding용
        "[UNK]": 1, # OOV(미등록 음소) 처리
        "|": 2,     # 단어 구분자 (여기선 음소 구분자 역할)
    }
    
    # 실제 음소 매핑 
    for i, phoneme in enumerate(vocab_list):
        vocab_dict[phoneme] = i + 3
        
    return vocab_dict

In [ ]:
vocab_dict = build_vocab(df)

In [ ]:
print("\n--- Vocab 검증 (반드시 2글자 이상 음소가 보여야 함) ---")
print(list(vocab_dict.keys())[:15])


--- Vocab 검증 (반드시 2글자 이상 음소가 보여야 함) ---
['[PAD]', '[UNK]', '|', 'AA', 'AE', 'AH', 'AO', 'AW', 'AX', 'AY', 'B', 'CH', 'D', 'DH', 'EH']


In [ ]:
# vocab.json 저장
with open("vocab.json", "w", encoding='utf-8') as f:
    json.dump(vocab_dict, f)

print(f"Vocab 생성 완료! 총 개수: {len(vocab_dict)}")

Vocab 생성 완료! 총 개수: 44


### 2) 토크나이저 설정
Wav2Vec2 기반 음성 모델 학습을 위해 CTC 토크나이저 + Feature Extractor를 결합한 Processor를 구성하였습니다.
1. Wav2Vec2CTCTokenizer 설정
    + 음소 문자열 → 정수 id 시퀀스로 변환
    + CTC 학습에 맞는 토큰 체계 구성
2. Wav2Vec2FeatureExtractor 설정
    + Raw waveform → 모델 입력 tensor 변환
    + 길이 padding 및 정규화 수행

In [ ]:
# 토크나이저 설정 (vocab.json 기반)
tokenizer = Wav2Vec2CTCTokenizer(
    "./vocab.json",     # 직접 생성한 음소 vocab
    unk_token="[UNK]",  # OOV 음소 처리
    pad_token="[PAD]",  # batch padding용
    word_delimiter_token="|",
    replace_word_delimiter_char=" " 
    # 모델 출력 → | → 디코딩 시 " " 로 바뀌도록 설정
)

# 피처 추출기 설정
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, 
    sampling_rate=16000, 
    padding_value=0.0, 
    do_normalize=True, 
    return_attention_mask=False
)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

## 3. 데이터 필터링
오디오 길이에 비해 정답이 너무 긴 데이터를 학습 전에 삭제하였습니다.

In [ ]:
# 데이터셋 생성
ds = Dataset.from_pandas(df)
ds = ds.cast_column("wav", Audio(sampling_rate=16000))

In [ ]:
def prepare_dataset(batch):
    audio = batch["wav"]
    batch["input_values"] = processor(audio["array"], sampling_rate=16000).input_values[0]
    with processor.as_target_processor():
        batch["labels"] = processor(batch["ppl"]).input_ids
    return batch

encoded_ds = ds.map(prepare_dataset, remove_columns=ds.column_names, num_proc=1)

Map:   0%|          | 0/3599 [00:00<?, ? examples/s]

/home/j-i14b104/.conda/envs/B104/lib/python3.9/site-packages/transformers/models/wav2vec2/processing_wav2vec2.py:180: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


In [ ]:
# 길이 검사 필터링
def filter_valid_data(batch):
    # Wav2Vec2는 입력 320개당 1개의 출력을 냅니다.
    # (입력 길이 / 320)이 정답(라벨) 길이보다 짧으면 학습 불가능 -> 버림
    audio_len = len(batch["input_values"])
    label_len = len(batch["labels"])
    return (audio_len // 320) > label_len

In [ ]:
print(f"필터링 전: {len(encoded_ds)}")
filtered_ds = encoded_ds.filter(filter_valid_data)
print(f"필터링 후: {len(filtered_ds)}") # 여기서 개수가 줄어들어야 정상입니다!

필터링 전: 3599


Filter:   0%|          | 0/3599 [00:00<?, ? examples/s]

필터링 후: 3599


In [ ]:
# Train/Validation/Test Split
# 1. 전체에서 Test 셋(10%) 분리 (최종 평가용, 학습 절대 X)
split_1 = filtered_ds.train_test_split(test_size=0.1, seed=42)
test_dataset = split_1["test"]
temp_dataset = split_1["train"]

# 2. 남은 데이터에서 Validation 셋(10%) 분리 (학습 중 감시용)
split_2 = temp_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_2["train"]
eval_dataset = split_2["test"]

print(f"\n데이터셋 분할 완료")
print(f"Train: {len(train_dataset)} (학습)")
print(f"Val  : {len(eval_dataset)} (검증)")
print(f"Test : {len(test_dataset)} (최종 평가 - 베이스 모델 비교용)")


데이터셋 분할 완료
Train: 2915 (학습)
Val  : 324 (검증)
Test : 360 (최종 평가 - 베이스 모델 비교용)


In [ ]:
train_dataset

Dataset({
    features: ['input_values', 'labels'],
    num_rows: 2915
})

In [ ]:
# df가 만들어진 상태에서 실행
print("--- 데이터 내부 확인 ---")
print(f"첫 번째 샘플: {df.iloc[0]['ppl']}")
# 예상 출력: "SIL F AO R ..."

--- 데이터 내부 확인 ---
첫 번째 샘플: SIL K L AH P S SIL AH N SIL B AO L S SIL AE N D S IH T IY Z G R UW T UW B IY SIL AO N L IY M EH M ER IY S SIL


## 4. 모델 설정
+ 베이스 모델: `facebook/wav2vec2-base-960h`: 영어 ASR로 사전학습된 Wav2Vec2를 가져와서 CTC head를 새로운 음소 vocab에 맞게 재구성하는 방식.
+ `vocab_size=len(processor.tokenizer)`: CTC 출력 레이어(Linear)가 새로운 vocab 크기만큼 나오도록 설정. 기존 960h 모델의 문자 vocab이 아니라 음소 vocab으로 바뀜.
+ `pad_token_id=...`: 라벨 패딩 처리할 때 PAD id 기준을 명확히 지정(Trainer/Collator에서 안정적).
+ `ignore_mismatched_sizes=True`: vocab 크기 변경 때문에 마지막 CTC head shape가 안 맞는데, 이 옵션으로 헤드는 새로 초기화하고 나머지 가중치는 그대로 로드.
+ `ctc_loss_reduction="mean"`: 배치 단위로 CTC loss를 평균내서 학습이 길이/샘플 수에 덜 민감하게끔 만듦.

In [ ]:
# 1. 모델 로드
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-base-960h",
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True
)


model.freeze_feature_encoder()          # 초반 레이어 고정
model.gradient_checkpointing_enable()
model.config.ctc_zero_infinity = True
model.config.mask_time_prob = 0.05 # SpecAugment의 time masking
model.config.mask_time_length = 10 # SpecAugment의 time masking
model.config.mask_feature_prob = 0.0
model.config.layerdrop = 0.05   # 학습 시 일부 Transformer layer를 drop시켜 정규화

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([32]) in the checkpoint and torch.Size([46]) in the model instantiated
- lm_head.weight: found shape torch.Size([32, 768]) in the checkpoint and torch.Size([46, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


평가 지표 (PER)$$PER = \frac{Substitutions + Deletions + Insertions}{Total Phonemes}$$

In [ ]:
from itertools import groupby

# Metric
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    # CTC 후처리: 중복 제거 + 특수토큰 제거
    def decode_tokens(ids):
        tokens = processor.tokenizer.convert_ids_to_tokens(ids)
        merged_tokens = [k for k, g in groupby(tokens)]
        valid_tokens = [t for t in merged_tokens if t not in ["[PAD]", "[UNK]", "|"]]
        
        return " ".join(valid_tokens)

    # 예측값과 정답값 모두 중복 제거 적용
    pred_str = [decode_tokens(ids) for ids in pred_ids]
    label_str = [decode_tokens(ids) for ids in pred.label_ids]

    # 로그 출력
    print(f"\n[중간 점검]")
    print(f"Ref : {label_str[0]}")
    print(f"Pred: {pred_str[0]}")

    per = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"per": per}

## 5. Data Collator 및 Trainer
Wav2Vec2 CTC 학습에서는 오디오 길이가 샘플마다 다르고 음소 라벨 길이도 샘플마다 다르기 때문에 입력과 라벨을 각각 다르게 padding 해야 합니다. 이를 위해 커스텀 DataCollator를 구현하였습니다.

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

## 6. Trainer 설정 및 실행

In [ ]:
training_args = TrainingArguments(
    output_dir="./wav2vec2-l2arctic-ft",
    group_by_length=True,
    per_device_train_batch_size=8, 
    gradient_accumulation_steps=4,
    eval_strategy="steps", # epoch 단위가 아니라 몇 step마다 평가/저장
    save_strategy="steps",
    learning_rate=3e-5, # 안정적인 학습률
    warmup_steps=500,
    lr_scheduler_type="linear",
    num_train_epochs=30,
    fp16=True,
    save_steps=100,
    eval_steps=100,
    logging_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="per",
    greater_is_better=False
)

In [ ]:
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics, 
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor.feature_extractor,
    
    # PER이 5번 연속으로 안 좋아지면 거기서 스톱!
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

/tmp/ipykernel_2035299/1129224544.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
print("학습 중...")
trainer.train()

학습 중...


Step,Training Loss,Validation Loss,Per
100,1.371100,0.943536,0.174953
200,1.351900,0.933911,0.172031
300,1.292700,0.904617,0.170457
400,1.288300,0.892741,0.163069
500,1.252800,0.857911,0.162041
600,1.168000,0.834254,0.159793
700,1.124500,0.802011,0.156099
800,1.069600,0.791093,0.155714
900,1.043300,0.757095,0.150061
1000,1.026200,0.743174,0.151153



[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D EH R IH G ER B ER R SIL IH T S B AO S AH M SIL AH N SIL AH S N AO R IY AH N S T IY M B OW T S SIL CH AE L EH N CH SIL D EH W IH L D ER N AH S T SIL

[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D EH R IH G ER B ER R SIL IH T S B AO S AH M SIL AH N SIL AH S N AO R IY AH N S T IY M B OW T S SIL CH AE L EH N CH SIL D EH W IH L D ER N AH S T SIL

[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D EH R IH G ER B EH ER R SIL IH T S B AO S AH M SIL AA N SIL AH S N AO R IY AH N S T IY M B OW T S SIL CH AE L AH N JH CH SIL D EH W IH L D ER N AH S T

TrainOutput(global_step=2760, training_loss=0.9935552527939064, metrics={'train_runtime': 3322.5585, 'train_samples_per_second': 26.32, 'train_steps_per_second': 0.831, 'total_flos': 2.9523397085762324e+18, 'train_loss': 0.9935552527939064, 'epoch': 30.0})

## 7. Best 모델 저장

In [ ]:
import os

# 1. 저장할 경로 지정
best_model_path = "./wav2vec2-l2arctic-ft-best"

# 2. 모델과 프로세서(토크나이저 포함) 저장
print(f"Best Model 저장 중... ({best_model_path})")
trainer.save_model(best_model_path)
processor.save_pretrained(best_model_path)

print("저장 완료!")

Best Model 저장 중... (./wav2vec2-l2arctic-ft-best)
저장 완료!


In [ ]:
# 실험 재현에 필요한 거의 모든 정보
print(training_args)

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=100,
eval_strategy=steps,
eval_use_gather_object=False,
fp16=True,
fp16_b

## 8. test data로 성능 테스트

In [ ]:
import torch
import evaluate
import numpy as np
from tqdm import tqdm
from itertools import groupby
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

# -------------------------------------------------------------------------
# 1. 저장된 Best Model 불러오기
# -------------------------------------------------------------------------
# 방금 저장하신 경로를 지정합니다.
saved_model_path = "./wav2vec2-l2arctic-ft-best"

print(f"모델 로드 중... ({saved_model_path})")
try:
    # 저장된 폴더에서 모델과 프로세서(토크나이저 설정 포함)를 가져옵니다.
    model = Wav2Vec2ForCTC.from_pretrained(saved_model_path)
    processor = Wav2Vec2Processor.from_pretrained(saved_model_path)
except OSError:
    print("경로에 모델이 없습니다. 방금 학습한 'model'과 'processor' 변수를 그대로 사용합니다.")
    # 만약 로드에 실패하면 현재 메모리에 있는 것을 씁니다.
    pass

# GPU 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# 평가 도구 로드
cer_metric = evaluate.load("cer")

# -------------------------------------------------------------------------
# 2. [핵심] 채점용 디코딩 함수 (SIL 및 특수 토큰 제거)
# -------------------------------------------------------------------------
def clean_decode_for_eval(ids, tokenizer):
    # 1. ID -> 토큰(문자) 변환
    tokens = tokenizer.convert_ids_to_tokens(ids)
    
    # 2. CTC 중복 제거 (Collapse): AA AA -> AA
    merged_tokens = [k for k, g in groupby(tokens)]
    
    # 3. 정제 (베이스 모델과 공정한 비교를 위해 SIL 제거)
    # [PAD], [UNK], |(구분자), <s>, </s>, SIL, SP 등을 모두 제외
    valid_tokens = [
        t for t in merged_tokens 
        if t not in ["[PAD]", "[UNK]", "|", "<s>", "</s>", "SIL", "SP"]
    ]
    
    return " ".join(valid_tokens)

# -------------------------------------------------------------------------
# 3. 추론 실행 (Test Dataset)
# -------------------------------------------------------------------------
print("\n--- 최종 추론 시작 (Test Dataset) ---")

predictions = []
references = []

# 학습 때 쓰지 않은 순수 Test 셋 사용
# (만약 변수명이 다르다면 dataset['test'] 등으로 바꿔주세요)
target_dataset = test_dataset 
batch_size = 16 

with torch.no_grad():
    for i in tqdm(range(0, len(target_dataset), batch_size)):
        # 배치핑 (Batch slicing)
        batch = target_dataset[i : i + batch_size]
        
        # 입력 데이터 처리 (Padding)
        inputs = processor(
            batch["input_values"], 
            sampling_rate=16000, 
            return_tensors="pt", 
            padding=True
        ).to(device)

        # 모델 예측 (Logits 산출)
        logits = model(**inputs).logits
        
        # 가장 높은 확률의 인덱스 추출 (Greedy Decoding)
        pred_ids = torch.argmax(logits, dim=-1)
        
        # -------------------------------------------------------
        # 디코딩 및 정제 (Predict)
        # -------------------------------------------------------
        pred_strs = [clean_decode_for_eval(ids.cpu().numpy(), processor.tokenizer) for ids in pred_ids]
        
        # -------------------------------------------------------
        # 정답 라벨 디코딩 (Reference)
        # -------------------------------------------------------
        # batch["labels"]가 리스트인지 텐서인지 확인하여 처리
        label_ids = batch["labels"]
        if isinstance(label_ids, torch.Tensor):
            label_ids = label_ids.cpu().numpy()
            
        label_strs = [clean_decode_for_eval(ids, processor.tokenizer) for ids in label_ids]
        
        predictions.extend(pred_strs)
        references.extend(label_strs)

# -------------------------------------------------------------------------
# 4. 최종 결과 출력
# -------------------------------------------------------------------------
final_per = cer_metric.compute(predictions=predictions, references=references)
baseline_per = 0.1263  # 베이스 모델 점수

print("\n" + "="*40)
print(f"="*40)
print(f"내 모델 PER : {final_per:.4f} ({final_per * 100:.2f}%)")
print(f"베이스 모델 PER: {baseline_per:.4f} (12.63%)")
print("-" * 40)

if final_per < baseline_per:
    gap = baseline_per - final_per
    print(f"베이스 모델보다 {gap:.4f} ({gap*100:.2f}%) 더 뛰어남")
else:
    print(f"차이: {final_per - baseline_per:.4f}")
print("="*40)

# 결과 샘플 3개 확인
print("\n[실제 예측 샘플 확인]")
for k in range(3):
    print(f"Ref : {references[k]}")
    print(f"Pred: {predictions[k]}")
    print("." * 30)

모델 로드 중... (./wav2vec2-l2arctic-ft-best)

--- 최종 추론 시작 (Test Dataset) ---


100%|██████████| 23/23 [00:11<00:00,  2.00it/s]


내 모델 PER : 0.1189 (11.89%)
베이스 모델 PER: 0.1263 (12.63%)
----------------------------------------
베이스 모델보다 0.0074 (0.74%) 더 뛰어남

[실제 예측 샘플 확인]
Ref : AY F AA L AO D DH AH L EY N AH V DH AH B R AH B AO Z AH L R EY L AO D L UH K IH NG G F AO R CH AE N S IH S
Pred: AY F AO L OW UH D DH AH L EY N AH V DH AH B R AH B OW Z AH L R IY L OW D L UH K IH NG F AO R CH AE N S IH S
..............................
Ref : T UW M AY S AH P R AY Z HH IY B IH G AE N T AH SH OW AE K SH AH L IH N T UW Z IY AE Z AH M IH N M AY F EY W AH
Pred: T UW M AY S AH P R AY Z HH IY B IH G AE N T UW SH OW AE K CH AH L EH N T UW Z IY AE Z AH M IH N M AY F EY V ER
..............................
Ref : D EH N AE N D AE S UW P ER HH IY T R AY D T AH F EY D AH M HH AH
Pred: D EH AE AE T D AH S UW P ER HH IY T R AY D T UW F EY T AH M HH AH
..............................


## 10. 추가 학습
현재 파인튜닝한 모델의 성능을 조금 더 올리고자 추가 학습을 진행하였습니다.   
추가 파인튜닝의 경우 이전 학습보다 조금 더 robust하게 만들기 위해 학습 강도를 올렸습니다.

In [44]:
import shutil
from transformers import TrainingArguments, Trainer, Wav2Vec2ForCTC, Wav2Vec2Processor, EarlyStoppingCallback

# -------------------------------------------------------------------------
# 1. 기존 Best Model 불러오기 (학습 시작점)
# -------------------------------------------------------------------------
# 아까 저장해둔 가장 성능 좋은 모델 경로
load_model_path = "./wav2vec2-l2arctic-ft-best" 

print(f"기존 Best Model 로드 중... ({load_model_path})")
model = Wav2Vec2ForCTC.from_pretrained(load_model_path)
processor = Wav2Vec2Processor.from_pretrained(load_model_path)

# 모델이 너무 편하게 공부하지 못하도록 마스킹을 강화합니다.
model.config.mask_time_prob = 0.08      # 0.05 -> 0.08 (시간축을 더 많이 가림)
model.config.mask_feature_prob = 0.02   # 0.01 -> 0.02 (주파수도 더 가림)
model.config.layerdrop = 0.1            # 0.05 -> 0.1 (신경망 10% 끄기)
model.config.ctc_zero_infinity = True   # 안전장치

# -------------------------------------------------------------------------
# 2. 추가 학습 설정 (Fine-tuning the Fine-tuned)
# -------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./wav2vec2-l2arctic-ft-ft", 
    group_by_length=True,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    save_strategy="steps",
    learning_rate=5e-5,  
    lr_scheduler_type="cosine",  # linear -> cosine (부드러운 하강)
    weight_decay=0.005,
    warmup_steps=500,    # 이미 학습된 모델이니 웜업은 짧게
    num_train_epochs=20, # 추가 20 에폭
    fp16=True,
    save_steps=100,
    eval_steps=100,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True, # 여전히 Best Model은 챙깁니다
    metric_for_best_model="per",
    greater_is_better=False
)

# -------------------------------------------------------------------------
# 3. 트레이너 실행
# -------------------------------------------------------------------------
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics, 
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor.feature_extractor,
    
    # 혹시 성능이 나빠지면 5번 만에 멈춤
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

기존 Best Model 로드 중... (./wav2vec2-l2arctic-ft-best)


/tmp/ipykernel_2035299/165351471.py:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [45]:
print("======현재 모델 성능 측정======")
baseline_metrics = trainer.evaluate()
baseline_per = baseline_metrics['eval_per']
print(f"기존 모델 PER: {baseline_per:.4f}")

print("======추가 학습======")
train_result = trainer.train()

print("======학습 후 성능 평가======")
final_metrics = trainer.evaluate()
final_per = final_metrics['eval_per']
print(f"추가 학습 후 PER: {final_per:.4f}")

======현재 모델 성능 측정======



[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D EH R IH G ER B EH R D SIL IH T S B AO S AH M SIL AA N SIL AH S N AO R IY IH N S T IY M B OW T S SIL CH AE L AH N JH SIL D AH W IH L D ER N AH S T SIL
기존 모델 PER: 0.1334
======추가 학습======


Step,Training Loss,Validation Loss,Model Preparation Time,Per
100,0.881800,0.684266,0.003200,0.134451
200,0.875700,0.685770,0.003200,0.133134
300,0.887700,0.672462,0.003200,0.136699
400,0.861300,0.679286,0.003200,0.134034
500,0.898600,0.681969,0.003200,0.139622
600,0.831000,0.669048,0.003200,0.131175
700,0.783700,0.671756,0.003200,0.134804
800,0.816900,0.667600,0.003200,0.135929
900,0.772400,0.668301,0.003200,0.128959
1000,0.769000,0.641840,0.003200,0.129987



[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D EH R IH G ER B EH R D SIL IH T S B AO S AH M SIL AA N SIL AH S N AO R IY AH N S T IY M B OW T S SIL CH AE L AH N JH SIL D AH W IH L D ER N AH S T SIL

[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D EH R IH G ER B ER R D T SIL IH T S B AO S AH M SIL AA N SIL AH S N AO R IY AH N S T IY M B OW T S SIL CH AE L AH N JH SIL D AH W IH L D ER N AH S T SIL

[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D AH R IH G ER B EH R D SIL IH T S B AO S AH M SIL AA N SIL S N AO R IY N S T IY M B OW T S SIL CH AE L AH N JH SIL D AH W IH L D ER N AH S T SIL


[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D EH R IH G ER B EH R D SIL IH T S B AO S AH M SIL AA N SIL S N AO R IY IH N S T IY M B OW T S SIL CH AE L AH N JH SIL D AH W IH L D ER N AH S T SIL
추가 학습 후 PER: 0.1270


In [46]:
final_save_path = "./wav2vec2-l2arctic-ft-ft-best"

if final_per < baseline_per:
    print(f"성능 향상 ({baseline_per:.4f} -> {final_per:.4f})")
    print(f"추가 학습된 모델을 저장합니다: {final_save_path}")
    trainer.save_model(final_save_path)
    processor.save_pretrained(final_save_path)
else:
    print(f"오히려 성능이 저하됌 ({baseline_per:.4f} -> {final_per:.4f})")
    print(f"기존 best 모델을 다시 복사해서 저장합니다.")
    
    # 기존 모델 다시 로드해서 저장
    model = Wav2Vec2ForCTC.from_pretrained(load_model_path)
    model.save_pretrained(final_save_path)
    processor.save_pretrained(final_save_path)

print(f"최종 모델 위치: '{final_save_path}'에 저장되었습니다.")

성능 향상 (0.1334 -> 0.1270)
추가 학습된 모델을 저장합니다: ./wav2vec2-l2arctic-ft-ft-best
최종 모델 위치: './wav2vec2-l2arctic-ft-ft-best'에 저장되었습니다.


### 추가 학습 한 번 더 진행

In [ ]:
import shutil
from transformers import TrainingArguments, Trainer, Wav2Vec2ForCTC, Wav2Vec2Processor, EarlyStoppingCallback

# -------------------------------------------------------------------------
# 1. 기존 Best Model 불러오기 (학습 시작점)
# -------------------------------------------------------------------------
# 아까 저장해둔 가장 성능 좋은 모델 경로
load_model_path = "./wav2vec2-l2arctic-ft-ft-best" 

print(f"기존 Best Model 로드 중... ({load_model_path})")
model = Wav2Vec2ForCTC.from_pretrained(load_model_path)
processor = Wav2Vec2Processor.from_pretrained(load_model_path)

# 모델이 너무 편하게 공부하지 못하도록 마스킹을 강화합니다.
model.config.mask_time_prob = 0.08      # 0.05 -> 0.08 (시간축을 더 많이 가림)
model.config.mask_feature_prob = 0.02   # 0.01 -> 0.02 (주파수도 더 가림)
model.config.layerdrop = 0.1            # 0.05 -> 0.1 (신경망 10% 끄기)
model.config.ctc_zero_infinity = True   # 안전장치

# -------------------------------------------------------------------------
# 2. 추가 학습 설정 (Fine-tuning the Fine-tuned)
# -------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./wav2vec2-l2arctic-ft-ft-ft", 
    group_by_length=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    save_strategy="steps",
    learning_rate=5e-5,  
    lr_scheduler_type="cosine",  # linear -> cosine (부드러운 하강)
    weight_decay=0.005,
    warmup_steps=500,    # 이미 학습된 모델이니 웜업은 짧게
    num_train_epochs=20, # 추가 20 에폭
    fp16=True,
    save_steps=100,
    eval_steps=100,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True, # 여전히 Best Model은 챙깁니다
    metric_for_best_model="per",
    greater_is_better=False
)

# -------------------------------------------------------------------------
# 3. 트레이너 실행
# -------------------------------------------------------------------------
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics, 
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor.feature_extractor,
    
    # 혹시 성능이 나빠지면 5번 만에 멈춤
    #callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

기존 Best Model 로드 중... (./wav2vec2-l2arctic-ft-ft-best)


/tmp/ipykernel_2035299/307065753.py:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
print("======현재 모델 성능 측정======")
baseline_metrics = trainer.evaluate()
baseline_per = baseline_metrics['eval_per']
print(f"기존 모델 PER: {baseline_per:.4f}")

print("======추가 학습======")
train_result = trainer.train()

print("======학습 후 성능 평가======")
final_metrics = trainer.evaluate()
final_per = final_metrics['eval_per']
print(f"추가 학습 후 PER: {final_per:.4f}")

======현재 모델 성능 측정======



[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D EH R IH G ER B EH R D SIL IH T S B AO S AH M SIL AA N SIL S N AO R IY IH N S T IY M B OW T S SIL CH AE L AH N JH SIL D AH W IH L D ER N AH S T SIL
기존 모델 PER: 0.1286
======추가 학습======


Step,Training Loss,Validation Loss,Model Preparation Time,Per
100,0.705500,0.635557,0.003300,0.134034
200,0.732500,0.628301,0.003300,0.130468
300,0.741500,0.639085,0.003300,0.132941
400,0.701800,0.658921,0.003300,0.134483
500,0.689400,0.641003,0.003300,0.128894
600,0.686000,0.645998,0.003300,0.131946
700,0.687800,0.664087,0.003300,0.133006
800,0.642400,0.655651,0.003300,0.133391
900,0.634500,0.637675,0.003300,0.135093
1000,0.625500,0.638413,0.003300,0.136121



[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D EH R IH G ER B EH R D SIL IH T S B AO S AH M SIL AA N SIL S N AO R IY AH N S T IY M B OW T S SIL CH AE L AH N JH SIL D AH W IH L D ER N AH S T SIL

[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D EH R IH G ER B EH R D SIL IH T S B AO S AH M SIL AA N SIL S N AO R IH N S T IY M B OW T S SIL CH AE L AH N JH SIL D AH W IH L D ER N AH S T SIL

[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D EH R IH G ER B EH R D SIL IH T S B AO S AH M SIL AA N SIL S N AO R IH N S T IY M B OW T S SIL CH AE L AH N JH SIL D AH W IH L D ER N AH S T SIL

[중간 점검]
R


[중간 점검]
Ref : SIL D AH R IH B ER B EH R D SIL IY T S B AO S AH M SIL AA N D SIL EH S N AO R T IH NG S T IY M B OW T S SIL CH AA L AH N JH D SIL D EH W IY L D ER N AH S SIL
Pred: SIL D EH R IH G ER L B EH R D SIL IH T S B AO S AH M SIL AA N SIL S N AO R IY N S T IY M B OW T S SIL CH AE L AH N JH SIL D AH W IH L D ER N AH S T SIL
추가 학습 후 PER: 0.1288


In [ ]:
final_save_path = "./wav2vec2-l2arctic-ft-ft-ft-best"

if final_per < baseline_per:
    print(f"성능 향상 ({baseline_per:.4f} -> {final_per:.4f})")
    print(f"추가 학습된 모델을 저장합니다: {final_save_path}")
    trainer.save_model(final_save_path)
    processor.save_pretrained(final_save_path)
else:
    print(f"오히려 성능이 저하됌 ({baseline_per:.4f} -> {final_per:.4f})")
    print(f"기존 best 모델을 다시 복사해서 저장합니다.")
    
    # 기존 모델 다시 로드해서 저장
    model = Wav2Vec2ForCTC.from_pretrained(load_model_path)
    model.save_pretrained(final_save_path)
    processor.save_pretrained(final_save_path)

print(f"최종 모델 위치: '{final_save_path}'에 저장되었습니다.")

오히려 성능이 저하됌 (0.1286 -> 0.1288)
기존 best 모델을 다시 복사해서 저장합니다.
최종 모델 위치: './wav2vec2-l2arctic-ft-ft-ft-best'에 저장되었습니다.


In [ ]:
import torch
import evaluate
import numpy as np
from tqdm import tqdm
from itertools import groupby
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

# -------------------------------------------------------------------------
# 1. 저장된 Best Model 불러오기
# -------------------------------------------------------------------------
# 방금 저장하신 경로를 지정합니다.
saved_model_path = "./wav2vec2-l2arctic-ft-ft-ft/checkpoint-1800"

print(f"모델 로드 중... ({saved_model_path})")
try:
    # 저장된 폴더에서 모델과 프로세서(토크나이저 설정 포함)를 가져옵니다.
    model = Wav2Vec2ForCTC.from_pretrained(saved_model_path)
    processor = Wav2Vec2Processor.from_pretrained(saved_model_path)
except OSError:
    print("경로에 모델이 없습니다. 방금 학습한 'model'과 'processor' 변수를 그대로 사용합니다.")
    # 만약 로드에 실패하면 현재 메모리에 있는 것을 씁니다.
    pass

# GPU 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# 평가 도구 로드
cer_metric = evaluate.load("cer")

# -------------------------------------------------------------------------
# 2. [핵심] 채점용 디코딩 함수 (SIL 및 특수 토큰 제거)
# -------------------------------------------------------------------------
def clean_decode_for_eval(ids, tokenizer):
    # 1. ID -> 토큰(문자) 변환
    tokens = tokenizer.convert_ids_to_tokens(ids)
    
    # 2. CTC 중복 제거 (Collapse): AA AA -> AA
    merged_tokens = [k for k, g in groupby(tokens)]
    
    # 3. 정제 (베이스 모델과 공정한 비교를 위해 SIL 제거)
    # [PAD], [UNK], |(구분자), <s>, </s>, SIL, SP 등을 모두 제외
    valid_tokens = [
        t for t in merged_tokens 
        if t not in ["[PAD]", "[UNK]", "|", "<s>", "</s>", "SIL", "SP"]
    ]
    
    return " ".join(valid_tokens)

# -------------------------------------------------------------------------
# 3. 추론 실행 (Test Dataset)
# -------------------------------------------------------------------------
print("\n--- 최종 추론 시작 (Test Dataset) ---")

predictions = []
references = []

# 학습 때 쓰지 않은 순수 Test 셋 사용
# (만약 변수명이 다르다면 dataset['test'] 등으로 바꿔주세요)
target_dataset = test_dataset 
batch_size = 16 

with torch.no_grad():
    for i in tqdm(range(0, len(target_dataset), batch_size)):
        # 배치핑 (Batch slicing)
        batch = target_dataset[i : i + batch_size]
        
        # 입력 데이터 처리 (Padding)
        inputs = processor(
            batch["input_values"], 
            sampling_rate=16000, 
            return_tensors="pt", 
            padding=True
        ).to(device)

        # 모델 예측 (Logits 산출)
        logits = model(**inputs).logits
        
        # 가장 높은 확률의 인덱스 추출 (Greedy Decoding)
        pred_ids = torch.argmax(logits, dim=-1)
        
        # -------------------------------------------------------
        # 디코딩 및 정제 (Predict)
        # -------------------------------------------------------
        pred_strs = [clean_decode_for_eval(ids.cpu().numpy(), processor.tokenizer) for ids in pred_ids]
        
        # -------------------------------------------------------
        # 정답 라벨 디코딩 (Reference)
        # -------------------------------------------------------
        # batch["labels"]가 리스트인지 텐서인지 확인하여 처리
        label_ids = batch["labels"]
        if isinstance(label_ids, torch.Tensor):
            label_ids = label_ids.cpu().numpy()
            
        label_strs = [clean_decode_for_eval(ids, processor.tokenizer) for ids in label_ids]
        
        predictions.extend(pred_strs)
        references.extend(label_strs)

# -------------------------------------------------------------------------
# 4. 최종 결과 출력
# -------------------------------------------------------------------------
final_per = cer_metric.compute(predictions=predictions, references=references)
baseline_per = 0.1263  # 베이스 모델 점수

print("\n" + "="*40)
print(f"="*40)
print(f"내 모델 PER : {final_per:.4f} ({final_per * 100:.2f}%)")
print(f"베이스 모델 PER: {baseline_per:.4f} (12.63%)")
print("-" * 40)

if final_per < baseline_per:
    gap = baseline_per - final_per
    print(f"베이스 모델보다 {gap:.4f} ({gap*100:.2f}%) 더 뛰어남")
else:
    print(f"차이: {final_per - baseline_per:.4f}")
print("="*40)

# 결과 샘플 3개 확인
print("\n[실제 예측 샘플 확인]")
for k in range(3):
    print(f"Ref : {references[k]}")
    print(f"Pred: {predictions[k]}")
    print("." * 30)

모델 로드 중... (./wav2vec2-l2arctic-ft-ft-ft/checkpoint-1800)

--- 최종 추론 시작 (Test Dataset) ---


100%|██████████| 23/23 [00:11<00:00,  2.02it/s]


내 모델 PER : 0.0941 (9.41%)
베이스 모델 PER: 0.1263 (12.63%)
----------------------------------------
베이스 모델보다 0.0322 (3.22%) 더 뛰어남

[실제 예측 샘플 확인]
Ref : AY F AA L AO D DH AH L EY N AH V DH AH B R AH B AO Z AH L R EY L AO D L UH K IH NG G F AO R CH AE N S IH S
Pred: AY F AA L OW D DH AH L EY N AH V DH AH B R AH B OW Z AH L R IY AH L OW D L UH K IH NG F AO R CH AE N S IH S
..............................
Ref : T UW M AY S AH P R AY Z HH IY B IH G AE N T AH SH OW AE K SH AH L IH N T UW Z IY AE Z AH M IH N M AY F EY W AH
Pred: T UW M AY S AH P R AY Z HH IY B IH G AE N T UW SH OW AE K CH AH L IH N T UW Z IY AE Z AH M IH N M AY F EY V ER
..............................
Ref : D EH N AE N D AE S UW P ER HH IY T R AY D T AH F EY D AH M HH AH
Pred: D EH N AE D AH S UH P ER HH IY T R AY D T AH F EY T AH M HH AH
..............................


In [ ]:
import torch
import librosa
import numpy as np
from itertools import groupby
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

# -------------------------------------------------------------------------
# 1. ARPAbet -> IPA 매핑 테이블 (이미지 기반 작성)
# -------------------------------------------------------------------------
arpabet_to_ipa = {
    "AA": "ɑ",  "AE": "æ",  "AH": "ʌ",  "AO": "ɔ",  "AW": "aʊ",
    "AX": "ə",  "AY": "aɪ", "B": "b",   "CH": "tʃ", "D": "d",
    "DH": "ð",  "EH": "ɛ",  "ER": "ɝ",  "EY": "eɪ", "F": "f",
    "G": "g",   "HH": "h",  "IH": "ɪ",  "IY": "i",  "JH": "dʒ",
    "K": "k",   "L": "l",   "M": "m",   "N": "n",   "NG": "ŋ",
    "OW": "oʊ", "OY": "ɔɪ", "P": "p",   "R": "ɹ",   "S": "s",
    "SH": "ʃ",  "T": "t",   "TH": "θ",  "UH": "ʊ",  "UW": "u",
    "V": "v",   "W": "w",   "Y": "j",   "Z": "z",   "ZH": "ʒ",
    "SIL": "",  "SP": ""   # 침묵은 표기하지 않음
}

# -------------------------------------------------------------------------
# 2. 모델 로드
# -------------------------------------------------------------------------
model_path = "./wav2vec2-l2arctic-ft-ft-ft/checkpoint-1800"
print(f"모델 로드 중... ({model_path})")

try:
    model = Wav2Vec2ForCTC.from_pretrained(model_path)
    processor = Wav2Vec2Processor.from_pretrained(model_path)
except OSError:
    print("모델 경로를 확인해주세요. 현재 메모리의 모델을 사용합니다.")
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# -------------------------------------------------------------------------
# 3. 추론 및 변환 함수
# -------------------------------------------------------------------------
def predict_my_voice_ipa(wav_file_path):
    print(f"\n 분석 및 변환 중: {wav_file_path}")
    
    # 1. 오디오 로드 (16000Hz)
    try:
        speech, rate = librosa.load(wav_file_path, sr=16000)
    except FileNotFoundError:
        print(" 파일을 찾을 수 없습니다.")
        return

    # 2. 모델 입력
    input_values = processor(
        speech, 
        sampling_rate=16000, 
        return_tensors="pt"
    ).input_values.to(device)

    # 3. 예측
    with torch.no_grad():
        logits = model(input_values).logits
    pred_ids = torch.argmax(logits, dim=-1)

    # 4. 토큰 변환 (ARPAbet)
    tokens = processor.tokenizer.convert_ids_to_tokens(pred_ids[0].cpu().numpy())
    
    # 5. 중복 제거 (Collapse)
    merged_tokens = [k for k, g in groupby(tokens)]
    
    # 6. 정제 (Clean ARPAbet)
    # 특수 토큰 제거
    clean_arpabet_list = [
        t for t in merged_tokens 
        if t not in ["[PAD]", "[UNK]", "|", "<s>", "</s>", "SIL", "SP"]
    ]
    
    # 7. [핵심] ARPAbet -> IPA 변환
    ipa_list = []
    for token in clean_arpabet_list:
        # 매핑 테이블에 있으면 변환, 없으면 그대로 출력(혹시 모를 에러 방지)
        ipa_char = arpabet_to_ipa.get(token, token)
        ipa_list.append(ipa_char)

    # 문자열로 합치기
    arpabet_output = " ".join(clean_arpabet_list)
    ipa_output = " ".join(ipa_list) # IPA는 보통 붙여서 쓰거나 좁은 간격

    # 결과 출력
    print("-" * 50)
    print(f"모델 예측 (ARPAbet): {arpabet_output}")
    print(f"변환 결과 (IPA)    : /{ipa_output}/")
    print("-" * 50)

# -------------------------------------------------------------------------
# 실행
# -------------------------------------------------------------------------
# 파일명을 넣어주세요
predict_my_voice_ipa("./test_wav/i_like_to_dance_test.wav")

모델 로드 중... (./wav2vec2-l2arctic-ft-ft-ft/checkpoint-1800)

 분석 및 변환 중: ./test_wav/i_like_to_dance_test.wav
--------------------------------------------------
모델 예측 (ARPAbet): AY L AY T G UH T AE N
변환 결과 (IPA)    : /aɪ l aɪ t g ʊ t æ n/
--------------------------------------------------
